# Módulo 06 · Aula 01 — Fundamentos de APIs

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O time do app precisa dos dados. Hoje eu exporto um CSV toda manhã e mando por e-mail. Eles pedem 'só o pedido 1042' e eu mando a planilha inteira. Ontem pediram para consultar estoque em tempo real e eu não tenho resposta."*
> — Você, na reunião de planejamento

O Atlas tem os dados. Falta uma **porta de entrada** para outros sistemas.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | HTTP de verdade | Métodos, status, cabeçalhos |
| 2 | REST | O que é (e o que não é) |
| 3 | Primeiro servidor | FastAPI + Uvicorn |
| 4 | Rotas e parâmetros | Path, query, ordem de declaração |
| 5 | `async` na prática | Quando usa e quando atrapalha |
| 6 | **OpenAPI** | Documentação que se escreve sozinha |
| 7 | Status codes | Escolher o certo importa |

## ⚙️ Como este notebook funciona

FastAPI é um **servidor**. Normalmente você o roda assim:

```bash
uvicorn main:app --reload
```

E o processo fica ocupado, servindo. Num notebook isso travaria o kernel.

A solução é o **`TestClient`**: ele executa a aplicação **em processo**, com semântica HTTP completa — status codes, cabeçalhos, validação, geração de OpenAPI — sem abrir porta nenhuma.

```python
cliente = TestClient(app)
resposta = cliente.get("/produtos/NB-01")
```

> 💡 **Isso não é uma gambiarra didática.** O `TestClient` é a forma padrão de **testar** APIs FastAPI — você vai reencontrá-lo no Módulo 07. Aprender com ele é aprender a ferramenta certa desde o começo.
>
> ▶️ **Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 06
# ═══════════════════════════════════════════════════════════════
import json
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


for _pacote, _mod in [("fastapi", "fastapi"), ("httpx", "httpx"),
                      ("uvicorn[standard]", "uvicorn"), ("pydantic", "pydantic")]:
    _garantir(_pacote, _mod)

import fastapi
from fastapi.testclient import TestClient

print(f"✅ FastAPI {fastapi.__version__}")
print(f"✅ Pydantic {__import__('pydantic').VERSION}")


# ═══════════════════════════════════════════════════════════════
#  Função auxiliar: exibe requisição e resposta lado a lado
# ═══════════════════════════════════════════════════════════════

def req(cliente, metodo: str, caminho: str, mostrar_corpo=True, **kwargs):
    """Faz uma requisição e imprime o resultado de forma legível."""
    resposta = getattr(cliente, metodo.lower())(caminho, **kwargs)

    cor = {2: "✅", 3: "↪️", 4: "⚠️", 5: "🔴"}.get(resposta.status_code // 100, "  ")
    print(f"{cor} {metodo.upper():<7} {caminho:<42} → {resposta.status_code}")

    if kwargs.get("json") is not None:
        corpo = json.dumps(kwargs["json"], ensure_ascii=False)
        print(f"   envio  : {corpo[:150]}{'...' if len(corpo) > 150 else ''}")

    if mostrar_corpo:
        try:
            dados = resposta.json()
            texto = json.dumps(dados, ensure_ascii=False, indent=2)
            linhas = texto.splitlines()
            for linha in linhas[:14]:
                print(f"   {linha}")
            if len(linhas) > 14:
                print(f"   ... (+{len(linhas) - 14} linhas)")
        except Exception:
            corpo = resposta.text[:200]
            if corpo.strip():
                print(f"   {corpo}")
    print()
    return resposta


print("✅ Função auxiliar `req(cliente, metodo, caminho, ...)` pronta")

## 1. HTTP — o protocolo por baixo

Toda API HTTP é uma conversa de **requisição** e **resposta**.

```
REQUISIÇÃO                          RESPOSTA
──────────                          ────────
GET /produtos/NB-01 HTTP/1.1        HTTP/1.1 200 OK
Host: api.aurora.com.br             Content-Type: application/json
Accept: application/json            Content-Length: 89
Authorization: Bearer eyJ...
                                    {"sku": "NB-01",
(corpo vazio no GET)                 "nome": "Notebook Dell",
                                     "preco": 2599.90}
```

### Os métodos

| Método | Faz | Idempotente? | Tem corpo? |
|--------|-----|:------------:|:----------:|
| `GET` | Lê | ✅ | ❌ |
| `POST` | Cria | ❌ | ✅ |
| `PUT` | Substitui **inteiro** | ✅ | ✅ |
| `PATCH` | Altera **parcialmente** | ❌ | ✅ |
| `DELETE` | Remove | ✅ | ❌ |

> 💡 **Idempotente** = repetir a mesma requisição produz o mesmo resultado. Chamar `DELETE /pedidos/1042` cinco vezes deixa o sistema no mesmo estado que chamar uma vez.
>
> **Por que importa:** se a rede cair depois de o servidor processar mas antes de a resposta chegar, o cliente vai tentar de novo. Num `POST` isso cria um pedido duplicado; num `PUT` não. É por isso que gateways de pagamento exigem uma *chave de idempotência* — você vai implementar uma no Módulo 07.

### Os status codes

| Faixa | Significa | Os que você mais usa |
|-------|-----------|----------------------|
| **2xx** | Deu certo | `200 OK`, `201 Created`, `204 No Content` |
| **3xx** | Redirecionamento | `301`, `304 Not Modified` |
| **4xx** | 🙋 **Erro do cliente** | `400`, `401`, `403`, `404`, `409`, `422`, `429` |
| **5xx** | 🔥 **Erro do servidor** | `500`, `502`, `503` |

> 🧭 **A distinção 4xx × 5xx é a mais importante.**
>
> - **4xx** — o cliente mandou algo errado. Ele precisa corrigir e tentar de novo.
> - **5xx** — **você** falhou. O cliente não pode fazer nada, e **isso deveria disparar um alerta**.
>
> Devolver `500` quando o usuário mandou um CPF inválido é ruído no seu monitoramento. Devolver `400` quando o banco caiu esconde um incidente.

| Code | Quando |
|------|--------|
| `200 OK` | Sucesso com corpo |
| `201 Created` | Recurso criado (devolva o `Location`) |
| `204 No Content` | Sucesso sem corpo (típico de `DELETE`) |
| `400 Bad Request` | Requisição malformada |
| `401 Unauthorized` | 🔑 Não autenticado (**quem é você?**) |
| `403 Forbidden` | 🚫 Autenticado, mas sem permissão (**você não pode**) |
| `404 Not Found` | Recurso não existe |
| `409 Conflict` | Conflito de estado (SKU duplicado) |
| `422 Unprocessable Entity` | Sintaxe ok, **semântica inválida** |
| `429 Too Many Requests` | Rate limit |

> ⚠️ **`401` vs `403` é a confusão mais comum.** `401` = não sei quem você é. `403` = sei quem você é e você não tem permissão. O nome do `401` ("Unauthorized") é infeliz — deveria ser "Unauthenticated".

## 2. REST — o que é e o que não é

**REST** é um estilo arquitetural. Na prática, o que importa:

| Princípio | Na prática |
|-----------|-----------|
| **Recursos, não ações** | `/pedidos/1042`, não `/buscarPedido?id=1042` |
| **Substantivos no plural** | `/produtos`, não `/produto` |
| **O método é o verbo** | `DELETE /pedidos/1042`, não `POST /deletarPedido` |
| **Sem estado** | Cada requisição carrega tudo que precisa |
| **Hierarquia** | `/pedidos/1042/itens` |

### O desenho de rotas do Atlas

| Método e rota | Faz | Status |
|---------------|-----|--------|
| `GET /produtos` | Lista, com filtros | `200` |
| `GET /produtos/{sku}` | Um produto | `200` / `404` |
| `POST /produtos` | Cria | `201` / `409` / `422` |
| `PUT /produtos/{sku}` | Substitui | `200` / `404` |
| `PATCH /produtos/{sku}` | Altera parcialmente | `200` / `404` |
| `DELETE /produtos/{sku}` | Remove | `204` / `404` |
| `GET /pedidos/{id}/itens` | Itens do pedido | `200` / `404` |

### ⚠️ Quando REST não cabe

Nem toda operação é CRUD. *"Cancelar pedido"* não é um `DELETE` — o pedido continua existindo, só muda de estado.

| Operação | Rota pragmática |
|----------|-----------------|
| Cancelar pedido | `POST /pedidos/1042/cancelamento` |
| Reprocessar carga | `POST /cargas/{id}/reprocessamento` |
| Gerar relatório | `POST /relatorios` → devolve `202 Accepted` + id |

> 🧭 **Pragmatismo acima de pureza.** Modelar a ação como um sub-recurso mantém o espírito REST sem forçar a barra. Ninguém ganha prêmio por REST perfeito; ganha-se por API que o cliente entende.

## 3. O primeiro servidor

FastAPI se apoia em três coisas que você já conhece:

| Peça | O que faz | Você viu em |
|------|-----------|-------------|
| **Type hints** | Definem entrada e saída | 04_04 |
| **Pydantic** | Valida e converte | 04_04 |
| **async/await** | Concorrência de I/O | 04_06 |

**Nada aqui é novo.** O FastAPI só junta as três de um jeito que gera documentação sozinho.

In [ ]:
from fastapi import FastAPI

app = FastAPI(
    title="Atlas API",
    description="API do sistema central da Aurora Comércio",
    version="1.0.0",
    contact={"name": "Engenharia Aurora", "email": "engenharia@aurora.com.br"},
)


@app.get("/")
def raiz():
    """Informações da API."""
    return {"servico": "Atlas API", "versao": "1.0.0", "documentacao": "/docs"}


@app.get("/saude")
def saude():
    """Healthcheck — usado pelo Docker e pelo balanceador de carga."""
    return {"status": "ok"}


cliente = TestClient(app)

req(cliente, "GET", "/")
req(cliente, "GET", "/saude")

### Rodando de verdade

```bash
# Salve o código acima em main.py, e então:
uvicorn main:app --reload --host 0.0.0.0 --port 8000

# ou, com o CLI do FastAPI:
fastapi dev main.py
```

| Flag | Faz |
|------|-----|
| `--reload` | Reinicia ao salvar o arquivo. **Só em desenvolvimento** |
| `--host 0.0.0.0` | Aceita conexões de fora do container |
| `--port 8000` | A porta |
| `--workers 4` | N processos. ⚠️ **Incompatível com `--reload`** |

Depois abra:

| URL | O que é |
|-----|---------|
| `http://localhost:8000/` | Sua API |
| `http://localhost:8000/docs` | 🎯 **Swagger UI** — interativo |
| `http://localhost:8000/redoc` | ReDoc — documentação para leitura |
| `http://localhost:8000/openapi.json` | O contrato, em JSON |

> ⚠️ **`--reload` em produção é um risco.** Ele monitora o sistema de arquivos, consome recursos e reinicia o processo por qualquer alteração. Em produção você usa Gunicorn com workers Uvicorn — assunto do Módulo 09.

## 4. Parâmetros de rota (*path*)

Tudo entre `{}` no caminho vira argumento da função. **A anotação de tipo faz a conversão e a validação.**

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI(title="Atlas API")

CATALOGO = {
    "NB-DELL-15": {"sku": "NB-DELL-15", "nome": "Notebook Dell Inspiron 15",
                   "categoria": "Notebooks", "preco": 2599.90, "estoque": 14},
    "MO-LG-24UW": {"sku": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide",
                   "categoria": "Monitores", "preco": 1199.00, "estoque": 31},
    "PE-LOG-M170": {"sku": "PE-LOG-M170", "nome": "Mouse Logitech M170",
                    "categoria": "Periféricos", "preco": 89.90, "estoque": 240},
}

PEDIDOS = {
    1042: {"id": 1042, "cliente": "Maria Souza", "status": "pago", "total": 5199.80},
    1043: {"id": 1043, "cliente": "João Lima", "status": "pendente", "total": 1199.00},
}


@app.get("/produtos/{sku}")
def buscar_produto(sku: str):
    """O `sku: str` vem da URL."""
    if sku not in CATALOGO:
        raise HTTPException(status_code=404, detail=f"Produto {sku} não encontrado")
    return CATALOGO[sku]


@app.get("/pedidos/{pedido_id}")
def buscar_pedido(pedido_id: int):
    """`pedido_id: int` — o FastAPI CONVERTE e VALIDA."""
    if pedido_id not in PEDIDOS:
        raise HTTPException(status_code=404, detail=f"Pedido {pedido_id} não encontrado")
    return PEDIDOS[pedido_id]


cliente = TestClient(app)

req(cliente, "GET", "/produtos/NB-DELL-15")
req(cliente, "GET", "/produtos/INEXISTENTE")
req(cliente, "GET", "/pedidos/1042")

In [ ]:
# 🎯 A anotação de tipo faz a validação — sem uma linha de código seu
req(cliente, "GET", "/pedidos/abc")

> 💡 **`422` com mensagem estruturada, de graça.** Você anotou `pedido_id: int` e ganhou: conversão, validação, mensagem de erro localizada no campo, e a documentação dizendo que o parâmetro é inteiro.
>
> É a mesma ideia da aula 04_04: **a anotação de tipo é a especificação**. Aqui ela finalmente *faz* alguma coisa em tempo de execução.

In [ ]:
# Validação com restrições: Path()
from fastapi import Path

app = FastAPI()


@app.get("/pedidos/{pedido_id}")
def buscar(pedido_id: int = Path(ge=1, le=999999, description="ID do pedido")):
    return {"id": pedido_id}


cliente = TestClient(app)
req(cliente, "GET", "/pedidos/1042")
req(cliente, "GET", "/pedidos/0")

### 🔴 A ordem das rotas importa

O FastAPI casa as rotas **na ordem em que foram declaradas**. Uma rota com parâmetro captura tudo — inclusive caminhos fixos declarados depois.

In [ ]:
app_errado = FastAPI()


@app_errado.get("/produtos/{sku}")
def buscar(sku: str):
    return {"rota": "dinâmica", "sku": sku}


@app_errado.get("/produtos/destaques")     # 🔴 nunca será alcançada
def destaques():
    return {"rota": "fixa", "produtos": ["NB-DELL-15"]}


c = TestClient(app_errado)
print("Rota fixa declarada DEPOIS da dinâmica:")
req(c, "GET", "/produtos/destaques")

In [ ]:
app_certo = FastAPI()


@app_certo.get("/produtos/destaques")      # ✅ fixa PRIMEIRO
def destaques():
    return {"rota": "fixa", "produtos": ["NB-DELL-15"]}


@app_certo.get("/produtos/{sku}")
def buscar(sku: str):
    return {"rota": "dinâmica", "sku": sku}


c = TestClient(app_certo)
print("Rota fixa declarada ANTES:")
req(c, "GET", "/produtos/destaques")
req(c, "GET", "/produtos/NB-DELL-15")

> 🧭 **A regra: rotas específicas antes das genéricas.**
>
> Este bug é traiçoeiro porque não gera erro nenhum — a rota simplesmente nunca é chamada, e o `sku` vira a string `"destaques"`. Você descobre quando alguém reclama que a página de destaques mostra "produto não encontrado".

## 5. Parâmetros de consulta (*query*)

Todo argumento da função que **não** está no caminho vira query parameter: `?limite=10&categoria=Notebooks`.

| Assinatura | Comportamento |
|------------|---------------|
| `limite: int` | **Obrigatório** |
| `limite: int = 10` | Opcional, com padrão |
| `categoria: str \| None = None` | Opcional, pode faltar |
| `limite: int = Query(10, ge=1, le=100)` | Com validação |

In [ ]:
from fastapi import Query

app = FastAPI(title="Atlas API")

PRODUTOS = [
    {"sku": "NB-DELL-15", "nome": "Notebook Dell Inspiron 15", "categoria": "Notebooks",   "preco": 2599.90},
    {"sku": "NB-ACER-N5", "nome": "Notebook Acer Nitro 5",     "categoria": "Notebooks",   "preco": 3299.00},
    {"sku": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide",   "categoria": "Monitores",   "preco": 1199.00},
    {"sku": "MO-SAM-ODY", "nome": "Monitor Samsung Odyssey",   "categoria": "Monitores",   "preco": 1849.00},
    {"sku": "PE-LOG-M170","nome": "Mouse Logitech M170",       "categoria": "Periféricos", "preco":   89.90},
    {"sku": "PE-RED-K552","nome": "Teclado Redragon K552",     "categoria": "Periféricos", "preco":  249.00},
]


@app.get("/produtos")
def listar(
    categoria: str | None = Query(None, description="Filtra por categoria"),
    preco_min: float | None = Query(None, ge=0),
    preco_max: float | None = Query(None, ge=0),
    busca: str | None = Query(None, min_length=2, description="Busca no nome"),
    ordenar: str = Query("nome", pattern="^(nome|preco|sku)$"),
    limite: int = Query(10, ge=1, le=100),
    pagina: int = Query(1, ge=1),
):
    """Listagem com filtros, ordenação e paginação."""
    resultado = PRODUTOS
    if categoria:
        resultado = [p for p in resultado if p["categoria"] == categoria]
    if preco_min is not None:
        resultado = [p for p in resultado if p["preco"] >= preco_min]
    if preco_max is not None:
        resultado = [p for p in resultado if p["preco"] <= preco_max]
    if busca:
        resultado = [p for p in resultado if busca.lower() in p["nome"].lower()]

    resultado = sorted(resultado, key=lambda p: p[ordenar])
    total = len(resultado)
    inicio = (pagina - 1) * limite

    return {
        "total": total,
        "pagina": pagina,
        "por_pagina": limite,
        "paginas": (total + limite - 1) // limite,
        "itens": resultado[inicio:inicio + limite],
    }


cliente = TestClient(app)

req(cliente, "GET", "/produtos?limite=2")
req(cliente, "GET", "/produtos?categoria=Notebooks&ordenar=preco")

In [ ]:
req(cliente, "GET", "/produtos?preco_min=100&preco_max=1500")
req(cliente, "GET", "/produtos?busca=logitech")

In [ ]:
# As validações em ação
print("── limite acima do máximo ──")
req(cliente, "GET", "/produtos?limite=500", mostrar_corpo=False)

print("── ordenar por campo não permitido ──")
r = req(cliente, "GET", "/produtos?ordenar=cor", mostrar_corpo=False)
print("   motivo:", r.json()["detail"][0]["msg"])

print("── busca curta demais ──")
r = req(cliente, "GET", "/produtos?busca=a", mostrar_corpo=False)
print("   motivo:", r.json()["detail"][0]["msg"])

> 💡 **`pattern="^(nome|preco|sku)$"` é uma lista branca.**
>
> Isso não é só validação — é **segurança**. Sem ela, `ordenar` viria direto do usuário e você o usaria para acessar um campo ou montar um `ORDER BY`. É exatamente o problema que você resolveu com lista branca no M03 (nome de tabela) e no M05 (filtro do Mongo).
>
> **Aqui o FastAPI faz a validação para você, e ainda documenta os valores aceitos.**

In [ ]:
# Listas em query: ?tag=gamer&tag=leve
app = FastAPI()


@app.get("/produtos")
def listar(tag: list[str] = Query(default=[]), skus: str | None = None):
    return {"tags_recebidas": tag, "skus": skus.split(",") if skus else []}


c = TestClient(app)
req(c, "GET", "/produtos?tag=gamer&tag=leve&tag=promo")
req(c, "GET", "/produtos?skus=NB-01,MO-01")

## 6. `def` ou `async def`?

Você viu na aula 04_06 que concorrência de I/O precisa de `async`. No FastAPI, a escolha é simples e tem uma armadilha.

| Você escreve | O FastAPI faz |
|--------------|---------------|
| `async def` | Roda no **event loop** |
| `def` | Roda num **threadpool** (não bloqueia o loop) |

> 🔴 **A armadilha:** se você declara `async def` e chama algo **bloqueante** lá dentro (`requests.get`, `time.sleep`, um driver de banco síncrono), você trava o event loop inteiro — e a API para de responder para **todos** os clientes.
>
> Escrever `def` é mais seguro: o FastAPI joga a função num thread e o loop segue livre.

### A regra de decisão

```
A função tem await lá dentro?
├── Sim  → async def
└── Não  → def
```

| Situação | Use |
|----------|-----|
| Chama `await httpx.AsyncClient(...)` | `async def` |
| Usa `requests` (síncrono) | **`def`** |
| Usa SQLAlchemy síncrono | **`def`** |
| Usa SQLAlchemy async | `async def` |
| Só processa dados em memória | `def` |
| CPU pesada | `def` (e considere fila — M07) |

In [ ]:
import time

app = FastAPI()


@app.get("/sincrona")
def sincrona():
    """Roda no threadpool — bloqueia a thread, não o loop."""
    time.sleep(0.05)
    return {"tipo": "def", "seguro": True}


@app.get("/assincrona")
async def assincrona():
    """Roda no event loop. Aqui NÃO pode haver bloqueio."""
    import asyncio
    await asyncio.sleep(0.05)     # ✅ await, não time.sleep
    return {"tipo": "async def", "seguro": True}


@app.get("/perigosa")
async def perigosa():
    """🔴 async def COM bloqueio — trava o loop inteiro."""
    time.sleep(0.05)              # 🔴 nunca faça isso
    return {"tipo": "async def com time.sleep", "seguro": False}


c = TestClient(app)
for rota in ["/sincrona", "/assincrona", "/perigosa"]:
    inicio = time.perf_counter()
    r = c.get(rota)
    print(f"{rota:<14} {r.status_code}  {(time.perf_counter()-inicio)*1000:>6.0f} ms  {r.json()}")

print("\n💭 Individualmente as três respondem igual.")
print("   A diferença aparece com CARGA: a /perigosa serializa todas as")
print("   requisições, porque cada uma bloqueia o event loop por 50 ms.")

## 7. 🎯 OpenAPI — a documentação que se escreve sozinha

Este é o argumento mais forte do FastAPI. A partir das suas anotações de tipo, ele gera o **contrato da API** no padrão OpenAPI 3.1.

In [ ]:
app = FastAPI(
    title="Atlas API",
    description="API do sistema central da Aurora Comércio.",
    version="1.0.0",
)


@app.get("/produtos/{sku}",
         summary="Busca um produto",
         description="Retorna os dados completos de um produto pelo SKU.",
         response_description="O produto encontrado",
         tags=["Produtos"])
def buscar_produto(
    sku: str = Path(description="SKU do produto", examples=["NB-DELL-15"]),
    incluir_estoque: bool = Query(True, description="Inclui a posição de estoque"),
):
    """A docstring também entra na documentação.

    Se houver `description=` no decorador, ela tem precedência.
    """
    return {"sku": sku, "estoque": 14 if incluir_estoque else None}


cliente = TestClient(app)
especificacao = cliente.get("/openapi.json").json()

print("OpenAPI:", especificacao["openapi"])
print("Título :", especificacao["info"]["title"], especificacao["info"]["version"])
print("Rotas  :", list(especificacao["paths"]))
print()

detalhe = especificacao["paths"]["/produtos/{sku}"]["get"]
print("summary   :", detalhe["summary"])
print("tags      :", detalhe["tags"])
print("parâmetros:")
for p in detalhe["parameters"]:
    obrig = "obrigatório" if p.get("required") else "opcional"
    print(f"   {p['name']:<18} em {p['in']:<6} ({obrig})  {p.get('description','')}")

In [ ]:
# As páginas de documentação existem e funcionam
for caminho, nome in [("/docs", "Swagger UI"), ("/redoc", "ReDoc"), ("/openapi.json", "Contrato")]:
    r = cliente.get(caminho)
    print(f"{nome:<12} {caminho:<16} → {r.status_code}  ({len(r.content):,} bytes)")

print("\n💡 Rodando com uvicorn, abra http://localhost:8000/docs")
print("   O Swagger UI permite EXECUTAR as rotas pelo navegador —")
print("   o time do app consegue explorar sua API sem escrever código.")

> 💭 **Por que isso muda o jogo.**
>
> Documentação de API tradicionalmente é um Word desatualizado. Aqui ela é **derivada do código** — se você renomear um parâmetro, a documentação muda junto. É impossível ficar desatualizada.
>
> E o `openapi.json` não é só documentação: é um **contrato legível por máquina**. Com ele você gera clientes em TypeScript, Java ou Python automaticamente, e valida contratos no CI.

## 8. Escolhendo o status code

In [ ]:
from fastapi import status, Response

app = FastAPI()
BANCO = {"NB-01": {"sku": "NB-01", "nome": "Notebook"}}


@app.post("/produtos", status_code=status.HTTP_201_CREATED)
def criar(sku: str, nome: str, resposta: Response):
    if sku in BANCO:
        raise HTTPException(status.HTTP_409_CONFLICT, f"SKU {sku} já existe")
    BANCO[sku] = {"sku": sku, "nome": nome}
    resposta.headers["Location"] = f"/produtos/{sku}"    # 📌 boa prática no 201
    return BANCO[sku]


@app.delete("/produtos/{sku}", status_code=status.HTTP_204_NO_CONTENT)
def remover(sku: str):
    if sku not in BANCO:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "não encontrado")
    del BANCO[sku]
    # 204 = sem corpo. Não retorne nada.


cliente = TestClient(app)

r = req(cliente, "POST", "/produtos?sku=MO-01&nome=Monitor")
print("   Location:", r.headers.get("location"))
print()
req(cliente, "POST", "/produtos?sku=MO-01&nome=Duplicado")
req(cliente, "DELETE", "/produtos/MO-01")
req(cliente, "DELETE", "/produtos/MO-01")

> 💡 **`from fastapi import status`** dá constantes nomeadas: `status.HTTP_201_CREATED` em vez de `201`. Mais legível e o editor autocompleta.
>
> 📌 **O cabeçalho `Location` no `201`** informa onde o recurso criado pode ser acessado. É parte da especificação HTTP e muitos clientes o utilizam.

## 🔧 Prática guiada — Primeira versão da API Atlas

In [ ]:
from fastapi import FastAPI, HTTPException, Query, Path, status

app = FastAPI(
    title="Atlas API",
    description="Sistema central da Aurora Comércio — produtos, pedidos e relatórios.",
    version="1.0.0",
    openapi_tags=[
        {"name": "Sistema", "description": "Saúde e informações"},
        {"name": "Produtos", "description": "Catálogo"},
        {"name": "Pedidos", "description": "Pedidos e itens"},
        {"name": "Relatórios", "description": "Agregações de negócio"},
    ],
)

# ── Dados em memória (o banco entra na aula 06_03) ──
CATALOGO = {
    "NB-DELL-15": {"sku": "NB-DELL-15", "nome": "Notebook Dell Inspiron 15",
                   "categoria": "Notebooks", "preco": 2599.90, "custo": 2120.00, "estoque": 14},
    "NB-ACER-N5": {"sku": "NB-ACER-N5", "nome": "Notebook Acer Nitro 5",
                   "categoria": "Notebooks", "preco": 3299.00, "custo": 2780.00, "estoque": 7},
    "MO-LG-24UW": {"sku": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide",
                   "categoria": "Monitores", "preco": 1199.00, "custo": 920.00, "estoque": 31},
    "PE-LOG-M170": {"sku": "PE-LOG-M170", "nome": "Mouse Logitech M170",
                    "categoria": "Periféricos", "preco": 89.90, "custo": 52.00, "estoque": 240},
    "PE-RED-K552": {"sku": "PE-RED-K552", "nome": "Teclado Redragon K552",
                    "categoria": "Periféricos", "preco": 249.00, "custo": 150.00, "estoque": 64},
}

PEDIDOS = {
    1042: {"id": 1042, "cliente": "Maria Souza", "cidade": "Campinas", "uf": "SP",
           "status": "pago", "canal": "site", "data": "2026-07-15",
           "itens": [{"sku": "NB-DELL-15", "qtd": 2, "preco_unitario": 2599.90},
                     {"sku": "MO-LG-24UW", "qtd": 1, "preco_unitario": 1199.00}]},
    1043: {"id": 1043, "cliente": "João Lima", "cidade": "São Paulo", "uf": "SP",
           "status": "pago", "canal": "app", "data": "2026-07-18",
           "itens": [{"sku": "PE-LOG-M170", "qtd": 10, "preco_unitario": 89.90}]},
    1044: {"id": 1044, "cliente": "Ana Costa", "cidade": "Curitiba", "uf": "PR",
           "status": "cancelado", "canal": "marketplace", "data": "2026-08-02",
           "itens": [{"sku": "NB-ACER-N5", "qtd": 1, "preco_unitario": 3299.00}]},
    1045: {"id": 1045, "cliente": "Maria Souza", "cidade": "Campinas", "uf": "SP",
           "status": "pago", "canal": "site", "data": "2026-08-05",
           "itens": [{"sku": "PE-RED-K552", "qtd": 3, "preco_unitario": 249.00},
                     {"sku": "MO-LG-24UW", "qtd": 2, "preco_unitario": 1199.00}]},
}


def total_do_pedido(pedido: dict) -> float:
    return round(sum(i["qtd"] * i["preco_unitario"] for i in pedido["itens"]), 2)


# ═══════════════════ Sistema ═══════════════════
@app.get("/saude", tags=["Sistema"], summary="Healthcheck")
def saude():
    return {"status": "ok", "produtos": len(CATALOGO), "pedidos": len(PEDIDOS)}


# ═══════════════════ Produtos ═══════════════════
@app.get("/produtos", tags=["Produtos"], summary="Lista produtos")
def listar_produtos(
    categoria: str | None = Query(None),
    preco_max: float | None = Query(None, ge=0),
    em_estoque: bool | None = Query(None, description="Só com estoque > 0"),
    ordenar: str = Query("nome", pattern="^(nome|preco|estoque|sku)$"),
    limite: int = Query(20, ge=1, le=100),
    pagina: int = Query(1, ge=1),
):
    itens = list(CATALOGO.values())
    if categoria:
        itens = [p for p in itens if p["categoria"] == categoria]
    if preco_max is not None:
        itens = [p for p in itens if p["preco"] <= preco_max]
    if em_estoque is not None:
        itens = [p for p in itens if (p["estoque"] > 0) == em_estoque]

    itens.sort(key=lambda p: p[ordenar])
    total = len(itens)
    inicio = (pagina - 1) * limite
    return {"total": total, "pagina": pagina, "por_pagina": limite,
            "itens": itens[inicio:inicio + limite]}


@app.get("/produtos/{sku}", tags=["Produtos"], summary="Busca um produto")
def buscar_produto(sku: str = Path(examples=["NB-DELL-15"])):
    if sku not in CATALOGO:
        raise HTTPException(status.HTTP_404_NOT_FOUND, f"Produto {sku} não encontrado")
    return CATALOGO[sku]


# ═══════════════════ Pedidos ═══════════════════
@app.get("/pedidos", tags=["Pedidos"], summary="Lista pedidos")
def listar_pedidos(
    status_pedido: str | None = Query(None, alias="status",
                                      pattern="^(pago|pendente|cancelado)$"),
    uf: str | None = Query(None, min_length=2, max_length=2),
    limite: int = Query(20, ge=1, le=100),
):
    itens = list(PEDIDOS.values())
    if status_pedido:
        itens = [p for p in itens if p["status"] == status_pedido]
    if uf:
        itens = [p for p in itens if p["uf"] == uf.upper()]
    resumo = [{**{k: v for k, v in p.items() if k != "itens"},
               "total": total_do_pedido(p), "qtd_itens": len(p["itens"])}
              for p in itens[:limite]]
    return {"total": len(itens), "itens": resumo}


@app.get("/pedidos/{pedido_id}", tags=["Pedidos"], summary="Busca um pedido")
def buscar_pedido(pedido_id: int = Path(ge=1)):
    if pedido_id not in PEDIDOS:
        raise HTTPException(status.HTTP_404_NOT_FOUND, f"Pedido {pedido_id} não encontrado")
    pedido = PEDIDOS[pedido_id]
    return {**pedido, "total": total_do_pedido(pedido)}


@app.get("/pedidos/{pedido_id}/itens", tags=["Pedidos"], summary="Itens do pedido")
def itens_do_pedido(pedido_id: int):
    if pedido_id not in PEDIDOS:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Pedido não encontrado")
    return [{**item,
             "nome": CATALOGO.get(item["sku"], {}).get("nome", "?"),
             "total": round(item["qtd"] * item["preco_unitario"], 2)}
            for item in PEDIDOS[pedido_id]["itens"]]


# ═══════════════════ Relatórios ═══════════════════
@app.get("/relatorios/faturamento-por-cidade", tags=["Relatórios"])
def faturamento_por_cidade():
    """O relatório do M01, agora exposto como recurso."""
    acumulado: dict[str, dict] = {}
    for pedido in PEDIDOS.values():
        if pedido["status"] != "pago":
            continue
        chave = f"{pedido['cidade']}/{pedido['uf']}"
        registro = acumulado.setdefault(chave, {"praca": chave, "pedidos": 0, "receita": 0.0})
        registro["pedidos"] += 1
        registro["receita"] = round(registro["receita"] + total_do_pedido(pedido), 2)

    linhas = sorted(acumulado.values(), key=lambda r: -r["receita"])
    total = round(sum(r["receita"] for r in linhas), 2)
    for r in linhas:
        r["share_pct"] = round(100 * r["receita"] / total, 1) if total else 0.0
    return {"total": total, "pracas": linhas}


cliente = TestClient(app)
print(f"✅ API com {len(app.routes)} rotas registradas\n")
req(cliente, "GET", "/saude")

In [ ]:
req(cliente, "GET", "/produtos?categoria=Periféricos&ordenar=preco")

In [ ]:
req(cliente, "GET", "/pedidos/1042")

In [ ]:
req(cliente, "GET", "/pedidos/1042/itens")

In [ ]:
req(cliente, "GET", "/pedidos?status=pago&uf=SP")

In [ ]:
req(cliente, "GET", "/relatorios/faturamento-por-cidade")

In [ ]:
# O contrato gerado
especificacao = cliente.get("/openapi.json").json()

print(f"{'MÉTODO':<8}{'ROTA':<44}{'TAG':<14}SUMMARY")
print("─" * 96)
for caminho, metodos in especificacao["paths"].items():
    for metodo, detalhe in metodos.items():
        tag = (detalhe.get("tags") or ["—"])[0]
        print(f"{metodo.upper():<8}{caminho:<44}{tag:<14}{detalhe.get('summary','')}")

> 💭 **Compare com a dor do início.** "Exporto um CSV toda manhã e mando por e-mail."
>
> Agora o time do app faz `GET /pedidos/1042` e recebe exatamente o que pediu, em JSON, na hora. E consegue **descobrir sozinho** o que a API oferece, abrindo `/docs`.
>
> Ainda falta muito: validação de entrada (06_02), banco de verdade (06_03) e autenticação (06_04). Mas o contrato já existe.

## 📝 Exercícios

**E1.** Para cada operação, escreva o método e a rota REST adequados: listar clientes; buscar cliente por id; criar cliente; atualizar e-mail do cliente; remover cliente; listar pedidos de um cliente; cancelar um pedido.

**E2.** Escolha o status code para cada situação: recurso criado; recurso não existe; e-mail já cadastrado; token expirado; usuário sem permissão; CPF com formato inválido; banco de dados fora do ar; remoção bem-sucedida.

**E3.** Explique com um exemplo prático a diferença entre `401` e `403`.

**E4.** Crie uma API com `/clientes` e `/clientes/{id}`, usando dados em memória. Trate o 404 corretamente.

**E5.** Reproduza o bug da ordem das rotas: crie `/produtos/{sku}` antes de `/produtos/mais-vendidos` e mostre o resultado. Depois corrija.

**E6.** Adicione a `/clientes` os filtros: `uf`, `segmento`, `busca` (no nome), com `ordenar` restrito por `pattern` e paginação.

**E7.** Crie uma rota que receba uma lista em query (`?uf=SP&uf=RJ`) e outra que receba valores separados por vírgula. Compare as duas abordagens.

**E8.** Escreva três rotas: uma `def`, uma `async def` correta e uma `async def` com bloqueio. Explique o que aconteceria com 100 requisições simultâneas em cada.

**E9.** Personalize o OpenAPI: título, descrição, versão, contato, licença e `openapi_tags` com descrição de cada grupo. Confirme lendo o `openapi.json`.

**E10.** Adicione o cabeçalho `Location` a uma rota `POST` que devolve `201`. Verifique com o `TestClient`.

**E11.** Crie `/relatorios/top-produtos` que agregue os itens de todos os pedidos pagos e devolva os N mais vendidos, com `N` vindo da query.

**E12.** Modele como recursos REST as operações: "cancelar pedido", "reprocessar carga" e "gerar relatório assíncrono". Justifique cada escolha.

In [ ]:
# E1 — sua resposta

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

## 📋 Cola de referência

```python
from fastapi import FastAPI, HTTPException, Query, Path, status, Response

app = FastAPI(title="Atlas API", version="1.0.0",
              openapi_tags=[{"name": "Produtos", "description": "..."}])

# ── Rotas ──
@app.get("/produtos", tags=["Produtos"], summary="Lista")
@app.post("/produtos", status_code=status.HTTP_201_CREATED)
@app.put("/produtos/{sku}")     # substitui inteiro
@app.patch("/produtos/{sku}")   # altera parcial
@app.delete("/produtos/{sku}", status_code=status.HTTP_204_NO_CONTENT)

# ⚠️ Rotas FIXAS antes das DINÂMICAS
@app.get("/produtos/destaques")   # primeiro
@app.get("/produtos/{sku}")       # depois

# ── Parâmetros ──
def rota(
    sku: str = Path(description="...", examples=["NB-01"]),      # da URL
    pedido_id: int = Path(ge=1, le=999999),
    limite: int = Query(10, ge=1, le=100),                       # ?limite=10
    busca: str | None = Query(None, min_length=2),
    ordenar: str = Query("nome", pattern="^(nome|preco)$"),      # 🔒 lista branca
    tags: list[str] = Query(default=[]),                         # ?tags=a&tags=b
    status_: str | None = Query(None, alias="status"),           # nome reservado
): ...

# ── Erros ──
raise HTTPException(status_code=404, detail="não encontrado")
raise HTTPException(status.HTTP_409_CONFLICT, "já existe")

# ── Resposta ──
def rota(resposta: Response):
    resposta.headers["Location"] = "/produtos/NB-01"
    resposta.status_code = 201

# ── def ou async def ──
# tem await dentro?  sim → async def   |   não → def
# 🔴 NUNCA: async def + time.sleep/requests/driver síncrono

# ── Testar ──
from fastapi.testclient import TestClient
cliente = TestClient(app)
r = cliente.get("/produtos?limite=5")
r.status_code   r.json()   r.headers

# ── Rodar ──
# uvicorn main:app --reload
# /docs  /redoc  /openapi.json
```

| Status | Quando |
|--------|--------|
| `200` | Sucesso com corpo |
| `201` | Criado (+ `Location`) |
| `204` | Sucesso sem corpo |
| `400` | Requisição malformada |
| `401` | Não autenticado |
| `403` | Sem permissão |
| `404` | Não existe |
| `409` | Conflito de estado |
| `422` | Validação falhou |
| `429` | Rate limit |
| `500` | 🔥 Erro **seu** — deve alertar |

## ✅ Checklist de saída

- [ ] Conheço os métodos HTTP e sei quais são idempotentes
- [ ] **Escolho o status code certo**, especialmente entre 4xx e 5xx
- [ ] Diferencio `401` de `403`
- [ ] Modelo rotas por recurso, não por ação
- [ ] Sei modelar operações que não cabem em CRUD
- [ ] Crio uma app FastAPI e a rodo com Uvicorn
- [ ] Sei que `--reload` não vai para produção
- [ ] Uso parâmetros de path com validação de tipo
- [ ] **Declaro rotas fixas antes das dinâmicas**
- [ ] Uso `Query()` com `ge`, `le`, `min_length` e `pattern`
- [ ] Uso `pattern` como lista branca de segurança
- [ ] **Sei decidir entre `def` e `async def`**
- [ ] Sei que `async def` com código bloqueante trava a API inteira
- [ ] Entendo o valor do OpenAPI gerado automaticamente
- [ ] Sei usar o `TestClient` para exercitar a API

---

### ➡️ Próxima aula

**`06_02_Pydantic_e_Rotas.ipynb`** — Corpo da requisição, validação, `response_model` e tratamento de erros. Onde a API para de aceitar qualquer coisa.